# Python Code Generator

## data preprocessing

In [2]:
from datasets import load_dataset

ds = load_dataset("flytech/python-codes-25k")
print(ds)

DatasetDict({
    train: Dataset({
        features: ['output', 'instruction', 'input', 'text'],
        num_rows: 49626
    })
})


In [3]:
from datasets import DatasetDict

def save_text_column(dataset_split, output_file):
    with open(output_file, "w", encoding="utf-8") as f:
        for text in dataset_split["text"]:

            f.write(text + "\n")

save_text_column(ds["train"], "train_text.txt")


print("Done!")

Done!


## defining model

In [4]:
# read it in to inspect it
with open('train_text.txt', 'r', encoding='utf-8') as f:
    text = f.read()
print("length of dataset in characters: ", len(text), "\n")
# let's look at the first 1000 characters
print(text[:1000])

length of dataset in characters:  24353616 

Help me set up my daily to-do list! Setting up your daily to-do list... ```python
tasks = []
while True:
    task = input('Enter a task or type 'done' to finish: ')
    if task == 'done': break
    tasks.append(task)
print(f'Your to-do list for today: {tasks}')
```
Create a shopping list based on my inputs! Creating a shopping list... ```python
shopping_list = {}
while True:
    item = input('Enter an item or type 'done' to finish: ')
    if item == 'done': break
    quantity = input(f'Enter the quantity for {item}: ')
    shopping_list[item] = quantity
print(f'Your shopping list: {shopping_list}')
```
Calculate how much time I spend on my phone per week! Calculating weekly phone usage... ```python
total_time = 0
for i in range(1, 8):
    time = float(input(f'Enter phone usage in hours for day {i}: '))
    total_time += time
print(f'You spend approximately {total_time} hours per week on your phone.')
```
Help me split the bill among my frien

In [7]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(f"Vocabulary size: {vocab_size}")

# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("def foo(x): return x + 1"))
print(decode(encode("print('hello world')")))

	
 !"#$%&'()*+,-./0123456789:;<=>?@ABCDEFGHIJKLMNOPQRSTUVWXYZ[\]^_`abcdefghijklmnopqrstuvwxyz{|}~
Vocabulary size: 98
[71, 72, 73, 3, 73, 82, 82, 11, 91, 12, 29, 3, 85, 72, 87, 88, 85, 81, 3, 91, 3, 14, 3, 20]
print('hello world')


In [8]:
# let's now encode the entire text dataset and store it into a torch.Tensor
import torch # we use PyTorch: https://pytorch.org
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000]) # the 1000 characters we looked at earier will to the GPT look like this

torch.Size([24353616]) torch.int64
tensor([43, 72, 79, 83,  3, 80, 72,  3, 86, 72, 87,  3, 88, 83,  3, 80, 92,  3,
        71, 68, 76, 79, 92,  3, 87, 82, 16, 71, 82,  3, 79, 76, 86, 87,  4,  3,
        54, 72, 87, 87, 76, 81, 74,  3, 88, 83,  3, 92, 82, 88, 85,  3, 71, 68,
        76, 79, 92,  3, 87, 82, 16, 71, 82,  3, 79, 76, 86, 87, 17, 17, 17,  3,
        67, 67, 67, 83, 92, 87, 75, 82, 81,  1, 87, 68, 86, 78, 86,  3, 32,  3,
        62, 64,  1, 90, 75, 76, 79, 72,  3, 55, 85, 88, 72, 29,  1,  3,  3,  3,
         3, 87, 68, 86, 78,  3, 32,  3, 76, 81, 83, 88, 87, 11, 10, 40, 81, 87,
        72, 85,  3, 68,  3, 87, 68, 86, 78,  3, 82, 85,  3, 87, 92, 83, 72,  3,
        10, 71, 82, 81, 72, 10,  3, 87, 82,  3, 73, 76, 81, 76, 86, 75, 29,  3,
        10, 12,  1,  3,  3,  3,  3, 76, 73,  3, 87, 68, 86, 78,  3, 32, 32,  3,
        10, 71, 82, 81, 72, 10, 29,  3, 69, 85, 72, 68, 78,  1,  3,  3,  3,  3,
        87, 68, 86, 78, 86, 17, 68, 83, 83, 72, 81, 71, 11, 87, 68, 86, 78, 12,
     

In [ ]:
# Let's now split up the data into train and validation sets
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [9]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 8 # how many independent sequences will we process in parallel?
block_size = 32 # what is the maximum context length for predictions?
max_iters = 500
eval_interval = 10
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 20
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0
# ------------

torch.manual_seed(1337)

# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a 2d batch{contains multiple independent sequences} of data of inputs x and targets y in shape (batch_size, block_size{num of tokens})
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix]) # result is (batch_size, block_size), notice that we have not reached to the tokens embedding stage yet, we are still working with the raw token indices
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval() # stop training
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train() # resume training
    return out

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x): # you can't fed batch{collection of independent sequences} to the transformer 
        B,T,C = x.shape
        k = self.key(x)   # (B,T,C) at this point C is the head_size, not the original embedding size because the embedding is changed to the attention space
        q = self.query(x) # (B,T,C)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape # B is the batch size, T is the sequence length (number of tokens in the context)

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C) # broadcasting happening
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = BigramLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))


0.213986 M parameters
step 0: train loss 4.7283, val loss 4.7521
step 10: train loss 3.7842, val loss 3.8012
step 20: train loss 3.4555, val loss 3.4373
step 30: train loss 3.4294, val loss 3.3433
step 40: train loss 3.4006, val loss 3.3030
step 50: train loss 3.3197, val loss 3.2638
step 60: train loss 3.2811, val loss 3.2613
step 70: train loss 3.2037, val loss 3.2463
step 80: train loss 3.2244, val loss 3.1399
step 90: train loss 3.1321, val loss 3.1324
step 100: train loss 3.1007, val loss 3.0802
step 110: train loss 3.0328, val loss 3.0253
step 120: train loss 3.0283, val loss 3.0980
step 130: train loss 2.9738, val loss 2.9752
step 140: train loss 2.9795, val loss 3.0324
step 150: train loss 2.9370, val loss 2.9381
step 160: train loss 2.8958, val loss 2.9460
step 170: train loss 2.8515, val loss 2.8963
step 180: train loss 2.8265, val loss 2.9194
step 190: train loss 2.8359, val loss 2.8728
step 200: train loss 2.8098, val loss 2.9057
step 210: train loss 2.8340, val loss 2.8810